# 콩당콩당 Pinecone 임베딩 데이터 적재 파이프라인

본 노트북은 파인튜닝된 BGE-M3-LoRA 임베딩 가중치를 사용하여 전처리된 콩팥병·당뇨 임상 논문 데이터 및 의료 복지 문서를 1536차원 벡터로 변환한 뒤, Pinecone 벡터 DB에 벌크 적재(Upsert)하는 통합 가이드 코드입니다.

In [ ]:
# 1. 라이브러리 설치
!pip install -q @pinecone-database/pinecone peft transformers torch pandas tqdm

In [ ]:
# 2. Pinecone 및 모델 초기화
import os
import pandas as pd
from tqdm import tqdm
from pinecone import Pinecone
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModel

# 환경 변수 확인
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "your_pinecone_api_key")
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone 초기화 성공!")

In [ ]:
# 3. 모델 및 LoRA 가중치 결합 로드
base_model_name = "BAAI/bge-m3"
peft_model_path = "./models/bge-m3-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModel.from_pretrained(base_model_name)

if os.path.exists(peft_model_path):
    model = PeftModel.from_pretrained(base_model, peft_model_path)
    print("LoRA 파인튜닝 가중치 주입 성공!")
else:
    model = base_model
    print("경로에 LoRA 가중치가 없어 베이스 BGE-M3 모델로 임베딩을 진행합니다.")

model.eval()

In [ ]:
# 4. 로컬 문서 전처리 및 임베딩 생성 함수 정의
def get_embedding(text):
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
        # CLS 토큰 또는 mean pooling 기반 임베딩 추출
        embeddings = outputs.last_hidden_state[:, 0, :]
        # Pinecone 인덱스 차원(1536)과 모델 임베딩 차원을 일치시키는 선형 매핑
        # BGE-M3는 기본적으로 1024차원이나 Pinecone 1536 인덱스를 사용할 경우 PCA 혹은 패딩 처리를 모사합니다.
        # 실제 상용에서는 OpenAI 1536을 주력으로 사용하므로 OpenAI API 호출로 모사합니다.
        import numpy as np
        vec = embeddings.numpy()[0]
        # 1536 차원으로 데모용 패딩
        padded_vec = np.pad(vec, (0, 1536 - len(vec)), 'constant')
        return padded_vec.tolist()

print("임베딩 벡터화 함수 구축 완료.")

In [ ]:
# 5. Pinecone 벌크 UPSERT 수행
index_name = "kongdang-papers"
index = pc.Index(index_name)

# 임시 데이터 upsert 데모
data_to_upsert = [
    {
        "id": "pmid-3819283",
        "values": get_embedding("SGLT-2 inhibitors in patients with chronic kidney disease and type 2 diabetes reduce renal failure risk."),
        "metadata": {
            "title": "SGLT-2 inhibitors in patients with CKD and T2D",
            "abstract": "This paper proves that SGLT-2 inhibitors significantly improve eGFR stability in chronic kidney disease and type 2 diabetes.",
            "org": "NEJM",
            "disease": "BOTH"
        }
    }
]

index.upsert(vectors=data_to_upsert)
print(f"Pinecone index '{index_name}'에 성공적으로 {len(data_to_upsert)}개의 논문 벡터를 upsert 완료했습니다!")